In [1]:
import pandas as pd

In [2]:
import json
import numpy as np

In [3]:
## ------------------------------------------------------------protein database parsing (esp evaluation col)------------------------------------
METRICS_SCHEMA = {
    "binding": {
        "valueType": "boolean",
        "dtype": "boolean",
    },
    "binding_strength": {
        "valueType": "label",
        "dtype": "string",
    },
    "expressed": {
        "valueType": "boolean",
        "dtype": "boolean",
    },
    "kd": {
        "valueType": "numeric",
        "dtype": "float",
    }}


df = pd.read_csv("proteinbase_all_data_28_10_2025.csv")

# Parse JSON array column
df["evaluations"] = df["evaluations"].apply(json.loads)


In [4]:
df.head()


,id,name,sequence,author,designMethod,evaluations
0,crimson-quail-cypress,pdgfrprot_16471,SHFVIGTAEAKSDSDEDIREALEKAANEAAEKAGLPPVKLTSVEIK...,mit,boltzgen,"[{'type': 'experimental', 'value': True, 'metr..."
1,steady-ant-oak,insulinprot_34946,NPVVEEARKLLEKAKELLDEARKLLEEGDYEKAKELIEEAEKLLKE...,mit,boltzgen,"[{'type': 'experimental', 'value': True, 'metr..."
2,noble-tiger-ember,pdgfrprot_35947,ITEEQRKELIEKAAELVVKAIEEGKLASEVKKELKEFAKKLGVELT...,mit,boltzgen,"[{'type': 'experimental', 'value': True, 'metr..."
3,green-crane-marble,insulinnano_52317,EVQLVESGGGLVQPGGSLRLSCAASGFTFSNYAMGWFRQAPGKGRE...,mit,boltzgen,"[{'type': 'experimental', 'value': True, 'metr..."
4,calm-quail-cypress,1g13prot_19735,GKLSGKQLLELFKEKVKKLLEGKEELTREEVLEIVEKAVEETVKEA...,mit,boltzgen,"[{'type': 'experimental', 'value': False, 'met..."


In [5]:
df=df.drop(["name","author", "designMethod"], axis=1)

In [6]:
def extract_metric_list(evaluations, metric_name, expected_value_type):
    """
    Returns a list of values for a given metric.
    If the metric is absent, returns np.nan (not an empty list).
    """
    values = [
        ev.get("value")
        for ev in evaluations
        if ev.get("metric") == metric_name
        and ev.get("valueType") == expected_value_type
    ]
    return values if values else np.nan


for metric, spec in METRICS_SCHEMA.items():
    df[metric] = df["evaluations"].apply(
        extract_metric_list,
        metric_name=metric,
        expected_value_type=spec["valueType"]
    )

In [7]:
df.head()

,id,sequence,evaluations,binding,binding_strength,expressed,kd
0,crimson-quail-cypress,SHFVIGTAEAKSDSDEDIREALEKAANEAAEKAGLPPVKLTSVEIK...,"[{'type': 'experimental', 'value': True, 'metr...","[False, False]","[None, None]","[True, True]",NaN
1,steady-ant-oak,NPVVEEARKLLEKAKELLDEARKLLEEGDYEKAKELIEEAEKLLKE...,"[{'type': 'experimental', 'value': True, 'metr...","[False, False]","[None, None]","[True, True]",NaN
2,noble-tiger-ember,ITEEQRKELIEKAAELVVKAIEEGKLASEVKKELKEFAKKLGVELT...,"[{'type': 'experimental', 'value': True, 'metr...","[True, True]","[Medium, Medium]","[True, True]","[3.37276852859295e-07, 3.67228685524738e-07]"
3,green-crane-marble,EVQLVESGGGLVQPGGSLRLSCAASGFTFSNYAMGWFRQAPGKGRE...,"[{'type': 'experimental', 'value': True, 'metr...","[False, False]","[None, None]","[True, True]",NaN
4,calm-quail-cypress,GKLSGKQLLELFKEKVKKLLEGKEELTREEVLEIVEKAVEETVKEA...,"[{'type': 'experimental', 'value': False, 'met...",[False],[None],[False],NaN


In [8]:
# handle target name
def extract_binding_target_list(evaluations):
    targets = [
        ev.get("target")
        for ev in evaluations
        if ev.get("metric") == "binding"
        and "target" in ev
    ]
    return targets if targets else np.nan

df["binding_target"] = df["evaluations"].apply(extract_binding_target_list)

In [9]:
df.head()

,id,sequence,evaluations,binding,binding_strength,expressed,kd,binding_target
0,crimson-quail-cypress,SHFVIGTAEAKSDSDEDIREALEKAANEAAEKAGLPPVKLTSVEIK...,"[{'type': 'experimental', 'value': True, 'metr...","[False, False]","[None, None]","[True, True]",NaN,"[human-pdgfr-beta, human-pdgfr-beta]"
1,steady-ant-oak,NPVVEEARKLLEKAKELLDEARKLLEEGDYEKAKELIEEAEKLLKE...,"[{'type': 'experimental', 'value': True, 'metr...","[False, False]","[None, None]","[True, True]",NaN,"[human-insulin-receptor, human-insulin-receptor]"
2,noble-tiger-ember,ITEEQRKELIEKAAELVVKAIEEGKLASEVKKELKEFAKKLGVELT...,"[{'type': 'experimental', 'value': True, 'metr...","[True, True]","[Medium, Medium]","[True, True]","[3.37276852859295e-07, 3.67228685524738e-07]","[human-pdgfr-beta, human-pdgfr-beta]"
3,green-crane-marble,EVQLVESGGGLVQPGGSLRLSCAASGFTFSNYAMGWFRQAPGKGRE...,"[{'type': 'experimental', 'value': True, 'metr...","[False, False]","[None, None]","[True, True]",NaN,"[human-insulin-receptor, human-insulin-receptor]"
4,calm-quail-cypress,GKLSGKQLLELFKEKVKKLLEGKEELTREEVLEIVEKAVEETVKEA...,"[{'type': 'experimental', 'value': False, 'met...",[False],[None],[False],NaN,[human-gm2a]


In [10]:
### --------------------------clean the dataset-----------------
# #Goal:  handle nan and multiple measurments:
# helpers:
def is_listlike(x):
    return isinstance(x, (list, tuple, np.ndarray))

def ensure_list(x):
    if pd.isna(x):
        return []
    return list(x) if is_listlike(x) else [x]

def all_equal(values):
    return len(set(values)) == 1 if values else True


In [11]:
def ensure_list(x):
    if is_listlike(x):
        return list(x)
    elif pd.isna(x):
        return []
    else:
        return [x]

# Ensure list-like consistency
for col in ["binding", "binding_target", "kd", "expressed"]:
    if col in df.columns:
        df[col] = df[col].apply(ensure_list)


# 1. Remove binders with no binding  or expression information at all
df = df[df["binding"].apply(len) > 0]
df = df[df["expressed"].apply(len) > 0]
df.head()

,id,sequence,evaluations,binding,binding_strength,expressed,kd,binding_target
0,crimson-quail-cypress,SHFVIGTAEAKSDSDEDIREALEKAANEAAEKAGLPPVKLTSVEIK...,"[{'type': 'experimental', 'value': True, 'metr...","[False, False]","[None, None]","[True, True]",[],"[human-pdgfr-beta, human-pdgfr-beta]"
1,steady-ant-oak,NPVVEEARKLLEKAKELLDEARKLLEEGDYEKAKELIEEAEKLLKE...,"[{'type': 'experimental', 'value': True, 'metr...","[False, False]","[None, None]","[True, True]",[],"[human-insulin-receptor, human-insulin-receptor]"
2,noble-tiger-ember,ITEEQRKELIEKAAELVVKAIEEGKLASEVKKELKEFAKKLGVELT...,"[{'type': 'experimental', 'value': True, 'metr...","[True, True]","[Medium, Medium]","[True, True]","[3.37276852859295e-07, 3.67228685524738e-07]","[human-pdgfr-beta, human-pdgfr-beta]"
3,green-crane-marble,EVQLVESGGGLVQPGGSLRLSCAASGFTFSNYAMGWFRQAPGKGRE...,"[{'type': 'experimental', 'value': True, 'metr...","[False, False]","[None, None]","[True, True]",[],"[human-insulin-receptor, human-insulin-receptor]"
4,calm-quail-cypress,GKLSGKQLLELFKEKVKKLLEGKEELTREEVLEIVEKAVEETVKEA...,"[{'type': 'experimental', 'value': False, 'met...",[False],[None],[False],[],[human-gm2a]


In [12]:
# 2. Remove binders that did not express
#    or expressed inconsistently (True & False)
def expressed_consistent_and_true(expr_list):
    return (
        len(expr_list) > 0
        and all_equal(expr_list)
        and expr_list[0] is True
    )

df = df[df["expressed"].apply(expressed_consistent_and_true)]

In [13]:
df

,id,sequence,evaluations,binding,binding_strength,expressed,kd,binding_target
0,crimson-quail-cypress,SHFVIGTAEAKSDSDEDIREALEKAANEAAEKAGLPPVKLTSVEIK...,"[{'type': 'experimental', 'value': True, 'metr...","[False, False]","[None, None]","[True, True]",[],"[human-pdgfr-beta, human-pdgfr-beta]"
1,steady-ant-oak,NPVVEEARKLLEKAKELLDEARKLLEEGDYEKAKELIEEAEKLLKE...,"[{'type': 'experimental', 'value': True, 'metr...","[False, False]","[None, None]","[True, True]",[],"[human-insulin-receptor, human-insulin-receptor]"
2,noble-tiger-ember,ITEEQRKELIEKAAELVVKAIEEGKLASEVKKELKEFAKKLGVELT...,"[{'type': 'experimental', 'value': True, 'metr...","[True, True]","[Medium, Medium]","[True, True]","[3.37276852859295e-07, 3.67228685524738e-07]","[human-pdgfr-beta, human-pdgfr-beta]"
3,green-crane-marble,EVQLVESGGGLVQPGGSLRLSCAASGFTFSNYAMGWFRQAPGKGRE...,"[{'type': 'experimental', 'value': True, 'metr...","[False, False]","[None, None]","[True, True]",[],"[human-insulin-receptor, human-insulin-receptor]"
5,dark-panther-jade,EVQLVESGGGLVQPGGSLRLSCAASSGAMSNYNMAWFRQAPGQERE...,"[{'type': 'experimental', 'value': True, 'metr...",[False],[None],[True],[],[human-gm2a]
...,...,...,...,...,...,...,...,...
1465,young-bear-cypress,MEPTDEEKRGKYVAKIVAEEALKAYTETKDEEELNFLLIKLAALSE...,"[{'type': 'experimental', 'value': True, 'metr...","[True, True]","[Medium, Medium]","[True, True]","[3.43827861164215e-07, 2.83451649384423e-07]","[il7r, il7r]"
1466,silent-ibis-onyx,KPLPSEQFADTFFDPNSEEAKFIREAYEVLKEAGAGESFEKAAAEY...,"[{'type': 'experimental', 'value': True, 'metr...","[False, False]","[None, None]","[True, True]",[],"[il7r, il7r]"
1467,violet-moth-crystal,GPLSKAEYLEEQIKFVEENKDKKELVKPKLIETLKYVGYDEEAEYY...,"[{'type': 'experimental', 'value': True, 'metr...","[True, True]","[Strong, Strong]","[True, True]","[1.69763114194875e-08, 1.66945877743878e-08]","[il7r, il7r]"
1468,steady-bat-crystal,EVNCELLNKLAEPVAKELFPENDLAKISTFYAALFFAYEGALKGKP...,"[{'type': 'experimental', 'value': True, 'metr...","[True, True]","[Weak, Weak]","[True, True]","[4.7655480131206e-06, 7.35208731797726e-06]","[pd-l1, pd-l1]"


In [14]:

# 3. Remove binders with inconsistent binding to the same target
#    Rule:
#    - either always binds (True only)
#    - or never binds (False only)
def binding_consistent(binding_list):
    return len(binding_list) > 0 and all_equal(binding_list)

len_before=len(df)
df_before=df.copy()
df = df[df["binding"].apply(binding_consistent)]
len_after=len(df)
print(f"Removed {len_before-len_after} rows")
#df
removed = df_before[df_before.id.isin(df.id) == False]
print(removed)



Removed 14 rows
                        id                                           sequence  \
526      scarlet-ibis-clay  RVKELEEEAKRKADEAEELKKRIDALQAKFNELLAAAKASSDPRKS...   
558     swift-otter-bronze                                    SSATAEADKRAAELA   
627      strong-quail-jade  DEEFRKKSHEVNQNYSDQILNLSTDIIETLEKVAKEAGLEDLADNN...   
706        vast-lion-cedar  DEVEKLREEAIKEIEEARRLIEEARERVGDACGPYADAAASSYEEA...   
717        rough-zebra-ice  MVEESPAELEDLEPLSQETFSDLWKLLPEEDASLMALTRAVEDLSS...   
722        jade-crane-moss  MPLNAFILFRQIRKQEGVVEGAEISKHSSVVKTFSDLWKLLPEKEK...   
723        quick-raven-ash  MPLLFSDLWKLLPRPARLADSLLDLARRVPGEAGRLEAVRAVRADA...   
728        pale-panda-fern  MLSDNVSRDDQLIFEIETLFRSGELSMSSRENGCGFSDLWKLLPPK...   
734         ivory-bee-fern  MIEALDVIEPELLIVVFTLCLANELPVVILTNEFSNKHQPENKEAF...   
736     lunar-panther-wave  MEEPQTDLSMEPMLSQETFSDLWKLLPERELQSFNPLKSIGDLLLP...   
738     crimson-bee-quartz  MEEPQLDLSIELLLSQETFSDLWKLLPENHLSSDLSMKAVDDLLLL...   
1411      si

In [15]:
df

,id,sequence,evaluations,binding,binding_strength,expressed,kd,binding_target
0,crimson-quail-cypress,SHFVIGTAEAKSDSDEDIREALEKAANEAAEKAGLPPVKLTSVEIK...,"[{'type': 'experimental', 'value': True, 'metr...","[False, False]","[None, None]","[True, True]",[],"[human-pdgfr-beta, human-pdgfr-beta]"
1,steady-ant-oak,NPVVEEARKLLEKAKELLDEARKLLEEGDYEKAKELIEEAEKLLKE...,"[{'type': 'experimental', 'value': True, 'metr...","[False, False]","[None, None]","[True, True]",[],"[human-insulin-receptor, human-insulin-receptor]"
2,noble-tiger-ember,ITEEQRKELIEKAAELVVKAIEEGKLASEVKKELKEFAKKLGVELT...,"[{'type': 'experimental', 'value': True, 'metr...","[True, True]","[Medium, Medium]","[True, True]","[3.37276852859295e-07, 3.67228685524738e-07]","[human-pdgfr-beta, human-pdgfr-beta]"
3,green-crane-marble,EVQLVESGGGLVQPGGSLRLSCAASGFTFSNYAMGWFRQAPGKGRE...,"[{'type': 'experimental', 'value': True, 'metr...","[False, False]","[None, None]","[True, True]",[],"[human-insulin-receptor, human-insulin-receptor]"
5,dark-panther-jade,EVQLVESGGGLVQPGGSLRLSCAASSGAMSNYNMAWFRQAPGQERE...,"[{'type': 'experimental', 'value': True, 'metr...",[False],[None],[True],[],[human-gm2a]
...,...,...,...,...,...,...,...,...
1465,young-bear-cypress,MEPTDEEKRGKYVAKIVAEEALKAYTETKDEEELNFLLIKLAALSE...,"[{'type': 'experimental', 'value': True, 'metr...","[True, True]","[Medium, Medium]","[True, True]","[3.43827861164215e-07, 2.83451649384423e-07]","[il7r, il7r]"
1466,silent-ibis-onyx,KPLPSEQFADTFFDPNSEEAKFIREAYEVLKEAGAGESFEKAAAEY...,"[{'type': 'experimental', 'value': True, 'metr...","[False, False]","[None, None]","[True, True]",[],"[il7r, il7r]"
1467,violet-moth-crystal,GPLSKAEYLEEQIKFVEENKDKKELVKPKLIETLKYVGYDEEAEYY...,"[{'type': 'experimental', 'value': True, 'metr...","[True, True]","[Strong, Strong]","[True, True]","[1.69763114194875e-08, 1.66945877743878e-08]","[il7r, il7r]"
1468,steady-bat-crystal,EVNCELLNKLAEPVAKELFPENDLAKISTFYAALFFAYEGALKGKP...,"[{'type': 'experimental', 'value': True, 'metr...","[True, True]","[Weak, Weak]","[True, True]","[4.7655480131206e-06, 7.35208731797726e-06]","[pd-l1, pd-l1]"


In [16]:
def reduce_row(row):
    out = row.copy()

    # binding
    out["binding"] = row["binding"][0] if row["binding"] else np.nan # np.nan case should not exist because of previous filtering

    # binding_target
    if row["binding_target"]:
        if not all_equal(row["binding_target"]):
            return None  # inconsistent targets → drop
        out["binding_target"] = row["binding_target"][0]
    else:
        out["binding_target"] = np.nan

    # kd
    out["kd"] = (
        float(np.mean(row["kd"])) if row["kd"] else np.nan
    )

    # expressed (already consistent & true by construction)
    out["expressed"] = True

    # binding strength
    # Filter out None values. If the list is empty after filtering, set np.nan.
    # Otherwise, take the first non-None value.
    valid_strengths = [s for s in row["binding_strength"] if s != "None"]
    if valid_strengths:
        out["binding_strength"] = valid_strengths[0]
    else:
        out["binding_strength"] = np.nan

    return out


df = df.apply(reduce_row, axis=1)
df = df[df['binding'].notna()]  # drop rows where binding is NaN

In [17]:
df

,id,sequence,evaluations,binding,binding_strength,expressed,kd,binding_target
0,crimson-quail-cypress,SHFVIGTAEAKSDSDEDIREALEKAANEAAEKAGLPPVKLTSVEIK...,"[{'type': 'experimental', 'value': True, 'metr...",False,NaN,True,NaN,human-pdgfr-beta
1,steady-ant-oak,NPVVEEARKLLEKAKELLDEARKLLEEGDYEKAKELIEEAEKLLKE...,"[{'type': 'experimental', 'value': True, 'metr...",False,NaN,True,NaN,human-insulin-receptor
2,noble-tiger-ember,ITEEQRKELIEKAAELVVKAIEEGKLASEVKKELKEFAKKLGVELT...,"[{'type': 'experimental', 'value': True, 'metr...",True,Medium,True,3.522528e-07,human-pdgfr-beta
3,green-crane-marble,EVQLVESGGGLVQPGGSLRLSCAASGFTFSNYAMGWFRQAPGKGRE...,"[{'type': 'experimental', 'value': True, 'metr...",False,NaN,True,NaN,human-insulin-receptor
5,dark-panther-jade,EVQLVESGGGLVQPGGSLRLSCAASSGAMSNYNMAWFRQAPGQERE...,"[{'type': 'experimental', 'value': True, 'metr...",False,NaN,True,NaN,human-gm2a
...,...,...,...,...,...,...,...,...
1465,young-bear-cypress,MEPTDEEKRGKYVAKIVAEEALKAYTETKDEEELNFLLIKLAALSE...,"[{'type': 'experimental', 'value': True, 'metr...",True,Medium,True,3.136398e-07,il7r
1466,silent-ibis-onyx,KPLPSEQFADTFFDPNSEEAKFIREAYEVLKEAGAGESFEKAAAEY...,"[{'type': 'experimental', 'value': True, 'metr...",False,NaN,True,NaN,il7r
1467,violet-moth-crystal,GPLSKAEYLEEQIKFVEENKDKKELVKPKLIETLKYVGYDEEAEYY...,"[{'type': 'experimental', 'value': True, 'metr...",True,Strong,True,1.683545e-08,il7r
1468,steady-bat-crystal,EVNCELLNKLAEPVAKELFPENDLAKISTFYAALFFAYEGALKGKP...,"[{'type': 'experimental', 'value': True, 'metr...",True,Weak,True,6.058818e-06,pd-l1


In [18]:
db =df.drop(["evaluations"], axis=1)

In [19]:
db

,id,sequence,binding,binding_strength,expressed,kd,binding_target
0,crimson-quail-cypress,SHFVIGTAEAKSDSDEDIREALEKAANEAAEKAGLPPVKLTSVEIK...,False,NaN,True,NaN,human-pdgfr-beta
1,steady-ant-oak,NPVVEEARKLLEKAKELLDEARKLLEEGDYEKAKELIEEAEKLLKE...,False,NaN,True,NaN,human-insulin-receptor
2,noble-tiger-ember,ITEEQRKELIEKAAELVVKAIEEGKLASEVKKELKEFAKKLGVELT...,True,Medium,True,3.522528e-07,human-pdgfr-beta
3,green-crane-marble,EVQLVESGGGLVQPGGSLRLSCAASGFTFSNYAMGWFRQAPGKGRE...,False,NaN,True,NaN,human-insulin-receptor
5,dark-panther-jade,EVQLVESGGGLVQPGGSLRLSCAASSGAMSNYNMAWFRQAPGQERE...,False,NaN,True,NaN,human-gm2a
...,...,...,...,...,...,...,...
1465,young-bear-cypress,MEPTDEEKRGKYVAKIVAEEALKAYTETKDEEELNFLLIKLAALSE...,True,Medium,True,3.136398e-07,il7r
1466,silent-ibis-onyx,KPLPSEQFADTFFDPNSEEAKFIREAYEVLKEAGAGESFEKAAAEY...,False,NaN,True,NaN,il7r
1467,violet-moth-crystal,GPLSKAEYLEEQIKFVEENKDKKELVKPKLIETLKYVGYDEEAEYY...,True,Strong,True,1.683545e-08,il7r
1468,steady-bat-crystal,EVNCELLNKLAEPVAKELFPENDLAKISTFYAALFFAYEGALKGKP...,True,Weak,True,6.058818e-06,pd-l1


In [20]:
db.to_csv("proteinbase_all_data_28_10_2025_cleaned.csv", index=False)